# Plato's Pizza : exploration des CSV
Auteur : Zohair

## Sommaire
1. Chargement des données
2. Structure des tables
3. Aperçu des données
4. Doublons et identifiants
5. Liens entre les tables
6. Prix et quantités
7. Période couverte (dates et heures)
8. Catégories, tailles et textes
9. Synthèse de l'exploration

## 1.Chargement des données 

In [4]:
import pandas as pd                                
orders = pd.read_csv("data/orders.csv")            
details = pd.read_csv("data/order_details.csv")    
pizzas = pd.read_csv("data/pizzas.csv")            
types = pd.read_csv("data/pizza_types.csv", encoding="cp1252")        
print(orders.shape, details.shape, pizzas.shape, types.shape)   

(21350, 3) (48620, 4) (96, 4) (32, 4)


## 2. Structure des tables 

In [5]:
orders.info()    
details.info()
pizzas.info()
types.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21350 entries, 0 to 21349
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   order_id  21350 non-null  int64 
 1   date      21350 non-null  object
 2   time      21350 non-null  object
dtypes: int64(1), object(2)
memory usage: 500.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   order_details_id  48620 non-null  int64 
 1   order_id          48620 non-null  int64 
 2   pizza_id          48620 non-null  object
 3   quantity          48620 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.5+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   pizza_id       96 non

### Observations
- orders : 21350 lignes, 3 colonnes, aucune valeur manquante. date et time sont lues comme du texte (object), il faudra les convertir pour analyser par mois, jour et heure.
- order_details : 48620 lignes, 4 colonnes, aucune valeur manquante.
- pizzas : 96 lignes, 4 colonnes, aucune valeur manquante, le prix est un nombre décimal.
- pizza_types : 32 lignes.

## 3. Aperçu des données

In [6]:
orders.head()

,order_id,date,time
0,1,2015-01-01,11:38:36
1,2,2015-01-01,11:57:40
2,3,2015-01-01,12:12:28
3,4,2015-01-01,12:16:31
4,5,2015-01-01,12:21:30


In [7]:
details.head()

,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1


In [8]:
pizzas.head()

,pizza_id,pizza_type_id,size,price
0,bbq_ckn_s,bbq_ckn,S,12.75
1,bbq_ckn_m,bbq_ckn,M,16.75
2,bbq_ckn_l,bbq_ckn,L,20.75
3,cali_ckn_s,cali_ckn,S,12.75
4,cali_ckn_m,cali_ckn,M,16.75


In [15]:
types.head()

,pizza_type_id,name,category,ingredients
0,bbq_ckn,The Barbecue Chicken Pizza,Chicken,"Barbecued Chicken, Red Peppers, Green Peppers,..."
1,cali_ckn,The California Chicken Pizza,Chicken,"Chicken, Artichoke, Spinach, Garlic, Jalapeno ..."
2,ckn_alfredo,The Chicken Alfredo Pizza,Chicken,"Chicken, Red Onions, Red Peppers, Mushrooms, A..."
3,ckn_pesto,The Chicken Pesto Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Spinach, Garli..."
4,southw_ckn,The Southwest Chicken Pizza,Chicken,"Chicken, Tomatoes, Red Peppers, Red Onions, Ja..."


### Observations
- orders : la date est au format 2015-01-01 et l'heure au format 11:38:36.
- order_details : une ligne = une pizza_id et sa quantité dans une commande. Une même commande peut avoir plusieurs lignes (order_id 2 apparaît 4 fois sur ces 5 lignes).
- pizzas : le pizza_id combine le type et la taille (bbq_ckn_s = bbq_ckn en taille S). Le prix dépend de la taille.
- pizza_types : les ingrédients sont tous dans une seule colonne texte, séparés par des virgules.

## 4. Doublons et identifiants

In [12]:
print(orders.duplicated().sum(),
details.duplicated().sum(),
pizzas.duplicated().sum(),
types.duplicated().sum())

0 0 0 0


In [13]:
print(orders["order_id"].is_unique,
details["order_details_id"].is_unique,
pizzas["pizza_id"].is_unique,
types["pizza_type_id"].is_unique)

True True True True


### Observations
- Aucune ligne en double dans les 4 tables (0 0 0 0).
- Les 4 identifiants (order_id, order_details_id, pizza_id, pizza_type_id) sont uniques (True) : ils serviront de clés primaires dans la base SQL.

## 5. Liens entre les tables

In [14]:
print(details["order_id"].isin(orders["order_id"]).all())          # chaque ligne de détail a-t-elle une commande ?
print(orders["order_id"].isin(details["order_id"]).all())          # chaque commande a-t-elle au moins une pizza ?
print(details["pizza_id"].isin(pizzas["pizza_id"]).all())          # chaque pizza vendue existe-t-elle dans pizzas ?
print(pizzas["pizza_type_id"].isin(types["pizza_type_id"]).all())  # chaque pizza a-t-elle un type ?

True
True
True
True


### Observations
Les 4 vérifications renvoient True : chaque ligne de détail a une commande, chaque commande a au moins une pizza, chaque pizza vendue existe dans pizzas et chaque pizza a un type. Les tables se relient sans perte, on pourra créer les clés étrangères dans la base SQL.

## 6. Prix et quantités

In [16]:
print(details["quantity"].value_counts())

quantity
1    47693
2      903
3       21
4        3
Name: count, dtype: int64


In [17]:
print(pizzas["price"].describe())

count    96.000000
mean     16.440625
std       4.090266
min       9.750000
25%      12.500000
50%      16.250000
75%      20.250000
max      35.950000
Name: price, dtype: float64


In [18]:
print(pizzas["size"].value_counts())

size
S      32
M      31
L      31
XL      1
XXL     1
Name: count, dtype: int64


In [19]:
print(pizzas[pizzas["size"].isin(["XL", "XXL"])])   # affiche seulement les pizzas en taille XL ou XXL

         pizza_id pizza_type_id size  price
42   the_greek_xl     the_greek   XL  25.50
43  the_greek_xxl     the_greek  XXL  35.95


In [20]:
print(pizzas["pizza_type_id"].value_counts())   # nombre de tailles proposées pour chaque type de pizza
print(details["pizza_id"].isin(["the_greek_xl", "the_greek_xxl"]).sum())   # nombre de lignes vendues en XL ou XXL

pizza_type_id
the_greek       5
bbq_ckn         3
peppr_salami    3
spinach_fet     3
spin_pesto      3
mexicana        3
mediterraneo    3
ital_veggie     3
green_garden    3
four_cheese     3
five_cheese     3
spinach_supr    3
spicy_ital      3
soppressata     3
sicilian        3
prsc_argla      3
ital_supr       3
cali_ckn        3
calabrese       3
pepperoni       3
pep_msh_pep     3
napolitana      3
ital_cpcllo     3
hawaiian        3
classic_dlx     3
big_meat        3
thai_ckn        3
southw_ckn      3
ckn_pesto       3
ckn_alfredo     3
veggie_veg      3
brie_carre      1
Name: count, dtype: int64
572


In [21]:
print(pizzas[pizzas["pizza_type_id"] == "brie_carre"])   # la seule ligne de la brie_carre, avec sa taille
print(details["pizza_id"].isin(["the_greek_xl", "the_greek_xxl"]).mean() * 100)   # % des lignes vendues en XL ou XXL

        pizza_id pizza_type_id size  price
44  brie_carre_s    brie_carre    S  23.65
1.1764705882352942


### Observations
- Quantités : 47693 lignes avec 1 pizza, 903 avec 2, 21 avec 3 et 3 avec 4. Aucune quantité nulle ou négative.
- Prix : de 9.75 à 35.95, médiane 16.25. Aucun prix nul ou négatif.
- Tailles : 30 types de pizzas existent en 3 tailles (S, M, L). Deux exceptions :
  - the_greek existe en 5 tailles, dont XL (25.50) et XXL (35.95). Le prix max vient donc de la the_greek XXL.
  - brie_carre n'existe qu'en taille S, à 23.65.
- C'est ce qui explique les comptes S 32, M 31, L 31, XL 1, XXL 1.
- Les grandes tailles ne sont pas des erreurs : 572 lignes vendues en XL ou XXL, soit 1.18 % des lignes.
- Décision : on garde tout, il n'y a rien à corriger dans les prix, les tailles ou les quantités.

## 7. Période couverte (dates et heures)

In [22]:
orders["date"] = pd.to_datetime(orders["date"])                               # transforme le texte en vraie date
print(orders["date"].min(), orders["date"].max(), orders["date"].nunique())   # 1er jour, dernier jour, nombre de jours avec des ventes
print(orders["time"].min(), orders["time"].max())                             # 1re et dernière heure de commande

2015-01-01 00:00:00 2015-12-31 00:00:00 358
09:52:21 23:05:52


In [23]:
jours_annee = pd.date_range("2015-01-01", "2015-12-31")    # tous les jours de 2015
print(jours_annee.difference(orders["date"]))              # les jours de 2015 sans aucune commande

DatetimeIndex(['2015-09-24', '2015-09-25', '2015-10-05', '2015-10-12',
               '2015-10-19', '2015-10-26', '2015-12-25'],
              dtype='datetime64[ns]', freq=None)


In [24]:
print(jours_annee.difference(orders["date"]).day_name())   # le nom du jour de la semaine pour chaque date sans commande

Index(['Thursday', 'Friday', 'Monday', 'Monday', 'Monday', 'Monday', 'Friday'], dtype='object')


### Observations
- Les données couvrent toute l'année 2015, du 2015-01-01 au 2015-12-31, avec 358 jours de ventes.
- 7 jours sans aucune commande :
  - le vendredi 25 décembre, probablement fermé pour Noël ;
  - le jeudi 24 et le vendredi 25 septembre ;
  - 4 lundis d'octobre (05, 12, 19 et 26).
  Ce sont probablement des jours de fermeture, à confirmer avec la direction. Ce ne sont pas des erreurs, on n'y touche pas.
- Les commandes vont de 09:52:21 à 23:05:52.
- Limite : une seule année de données, donc pas de comparaison d'une année sur l'autre.

## 8. Catégories et textes

In [25]:
print(types["category"].value_counts())                                   # nombre de types de pizzas par catégorie
print(types.loc[types["name"].str.contains(r"[^\x00-\x7F]"), "name"])    # noms avec un caractère spécial (accent, apostrophe...)
print(types.loc[types["ingredients"].str.contains(r"[^\x00-\x7F]"), "ingredients"])   # pareil pour les ingrédients

category
Supreme    9
Veggie     9
Classic    8
Chicken    6
Name: count, dtype: int64
Series([], Name: name, dtype: object)
15    ‘Nduja Salami, Pancetta, Tomatoes, Red Onions,...
Name: ingredients, dtype: object


### Observations
- 4 catégories : Supreme 9 types, Veggie 9, Classic 8, Chicken 6.
- Aucun nom de pizza ne contient de caractère spécial (Series vide).
- Un seul ingrédient en contient : ‘Nduja Salami (ligne 15), avec une apostrophe typographique. C'est elle qui obligeait à lire le fichier en cp1252.
- Décision : on garde le texte tel quel. Lu en cp1252, il s'affiche correctement et sera enregistré proprement dans la base.

## 9. Synthèse de l'exploration

| Point vérifié | Ce qu'on a vu | Décision |
|---|---|---|
| Valeurs manquantes | Aucune dans les 4 tables | Rien à faire |
| Doublons | 0 ligne en double, 4 identifiants uniques | Ils deviennent les clés primaires |
| Liens entre tables | 4 vérifications à True | Clés étrangères possibles, jointures sans perte |
| Quantités et prix | Quantités de 1 à 4, prix de 9.75 à 35.95 | Rien à corriger |
| Tailles | the_greek en 5 tailles (XL, XXL), brie_carre seulement en S | Cas réels, on garde (572 lignes en XL/XXL) |
| Dates | Toute l'année 2015, 358 jours de ventes, 7 jours sans commande | Probablement des fermetures, on garde |
| Types | date et time en texte | Convertir dans les analyses |
| Encodage | pizza_types en cp1252, 1 ingrédient touché (‘Nduja) | Lire avec encoding="cp1252" |

Conclusion : les données sont propres. Aucune ligne n'est supprimée, on peut construire la base SQL directement.